In [ ]:
import pandas as pd
import numpy as np

snapshot: date when AACT data was downloaded

In [ ]:
snapshot = pd.Timestamp('2026-09-12')

date_cols: list of column names that contain dates

In [ ]:
date_cols = ['start_date',
             'completion_date',
             'primary_completion_date',
             'study_first_submitted_date',
             'last_update_posted_date',
             'end_date']

Import csv file<br>

In [ ]:
raw = pd.read_csv("clinical_trials_master_table.csv", parse_dates = date_cols, low_memory = False)
print(raw.shape)
print(raw.head())

(115328, 29)
        nct_id overall_status   phase start_date start_date_type  \
0  NCT00000113      COMPLETED  PHASE3 1997-09-30             NaN   
1  NCT00000114      COMPLETED  PHASE3 1984-05-31             NaN   
2  NCT00000115      COMPLETED  PHASE2 1990-12-31             NaN   
3  NCT00000116      COMPLETED  PHASE3 1996-05-31          ACTUAL   
4  NCT00000117      COMPLETED  PHASE3 1995-08-31             NaN   

  completion_date completion_date_type primary_completion_date  \
0      2013-09-30               ACTUAL              2001-10-31   
1      1987-06-30               ACTUAL                     NaT   
2      1994-06-30               ACTUAL                     NaT   
3      2002-09-30               ACTUAL              2002-09-30   
4      1997-12-31               ACTUAL                     NaT   

  primary_completion_date_type study_first_submitted_date  ... number_of_arms  \
0                       ACTUAL                 1999-09-23  ...            2.0   
1                  

# Check the raw table for problems

1. Check that each trial has a unique nct_id
2. Count the number of trials with each status (overall_status)
3. Count the number of trials with each phase (phase)
4. Count the number of trials with each sponsor type (agency_class)

Check: count how many missing entries there are for each column in the master table


In [ ]:
assert raw['nct_id'].is_unique, 'duplicate trials'
display(raw['overall_status'].value_counts(dropna=False))
display(raw['phase'].value_counts(dropna=False))
display(raw['agency_class'].value_counts(dropna=False))

raw.isna().sum().sort_values(ascending=False).to_frame('n_missing')

,count
overall_status,
COMPLETED,64006
UNKNOWN,14078
TERMINATED,11406
RECRUITING,11147
ACTIVE_NOT_RECRUITING,5446
NOT_YET_RECRUITING,4413
WITHDRAWN,4094
SUSPENDED,377
ENROLLING_BY_INVITATION,361


,count
phase,
PHASE2,65493
PHASE3,42245
PHASE2/PHASE3,7590


,count
agency_class,
OTHER,58952
INDUSTRY,47359
NIH,3855
OTHER_GOV,2192
NETWORK,2091
FED,713
INDIV,126
UNKNOWN,40


,n_missing
why_stopped,101514
is_fda_regulated_drug,59765
start_date_type,50574
allocation,28234
has_dmc,24132
duration_days,23358
end_date_type,22585
end_date,22585
number_of_arms,10861
has_us_facility,10621


# Table containing overall_status (row) and end_date_type (col)

Create a count table with one variable as rows (overall_status) and another variables as cols (end_date_type) with blank values replaced with NaN <br>
<br>
SNAPSHOT column: means the trial hasn't ended yet, so its end date is set to the downloaded date '09-12-2026'

In [ ]:
pd.crosstab(raw['overall_status'], raw['end_date_type'].fillna('NaN'))

end_date_type,ACTUAL,ESTIMATED,MISSING,NaN,SNAPSHOT
overall_status,,,,,
ACTIVE_NOT_RECRUITING,0,0,0,0,5446
COMPLETED,59695,0,4311,0,0
ENROLLING_BY_INVITATION,0,0,0,0,361
NOT_YET_RECRUITING,0,0,0,4413,0
RECRUITING,0,0,0,0,11147
SUSPENDED,0,0,0,0,377
TERMINATED,10701,178,527,0,0
UNKNOWN,0,0,0,14078,0
WITHDRAWN,0,0,0,4094,0


# Exclusion: Throw out trials that can't be analyzed

Why remove trials: Survival analysis needs 3 things:
1. When it started
2. When it ended (or snapshot date)
3. How it ended <br>

Some trials are missing 1+ of these or have impossible values (for ex. no start date, end date before start date, unknown status). <br>

exclude() function: remove the rows and log the count<br>

Purpose: to turn the raw table (csv) into a usable set of trials

log: creates a list holding an entry; each entry is a dictionary, a set of labeled values:
1. step number
2. reason
3. how many were removed
4. how many remain

exclude(): a helper function given 4 things:
1. df: current table
2. mask: True/False col where True means 'remove this row'
3. step: step number
4. reason: short description for the log




In [ ]:
log = [{'step': '0',
        'reason': 'Master table (SQL population)',
        'n_removed': 0,
        'n_remaining': len(raw)}]

def exclude(df, mask, step, reason):
  mask = mask.fillna(False)
  n = int(mask.sum())
  log.append({'step': str(step),
              'reason': reason,
              'n_removed': n,
              'n_remaining': len(df) -  n})
  return df.loc[~mask].copy()

# Exclusions

1. Unknown status (unverified) and not yet recruiting
2. Missing start date or start date after the snapshot date
3. Missing enrollment
4. Completed with zero enrollment
5. End date on or before start date
6. Sponsor class individual or unknown

After these exclusions, there are 90282 unique trials remaining.

In [ ]:
df = raw.copy()

# Reset the log to avoid duplicate entries when re-running the cell
log = [{'step': '0',
        'reason': 'Master table (SQL population)',
        'n_removed': 0,
        'n_remaining': len(raw)}]

withdrawn = df[df['overall_status'] == 'WITHDRAWN'].copy()
df = exclude(df,
            df['overall_status'] == 'WITHDRAWN',
            '0b',
            'Withdrawn (set aside for sub-question 8)')

df = exclude(df,
             df['overall_status'].isin(['UNKNOWN', 'NOT_YET_RECRUITING']),
             1,
             'Unknown status or not yet recruiting')

df = exclude(df,
             df['start_date'].isna() | (df['start_date'] > snapshot),
             2,
             'Missing start date or start date after snapshot date')

df = exclude(df,
             df['enrollment'].isna(),
             3,
             'Missing enrollment')

df = exclude(df,
             (df['overall_status'] == 'COMPLETED') & (df["enrollment"] == 0),
             4,
             'Completed with zero enrollment')

df = exclude(df,
             df['end_date'] <= df['start_date'],
             5,
             'End date on or before start date')

df = exclude(df,
             df['agency_class'].isin(['INDIV', 'UNKNOWN']),
             6,
             'Individual or unknown sponsor')

pd.DataFrame(log)

,step,reason,n_removed,n_remaining
0,0,Master table (SQL population),0,115328
1,0b,Withdrawn (set aside for sub-question 8),4094,111234
2,1,Unknown status or not yet recruiting,18491,92743
3,2,Missing start date or start date after snapsho...,881,91862
4,3,Missing enrollment,1324,90538
5,4,Completed with zero enrollment,79,90459
6,5,End date on or before start date,68,90391
7,6,Individual or unknown sponsor,109,90282


# Inspect long durations before choosing a cutoff

Calculate how long each trial lasts and show the longest ones to decide whether some durations are too long to be believable.<br>


In [ ]:
df['duration_years'] = (df['end_date'] - df['start_date']).dt.days / 365
display(df['duration_years'].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))
display(df.groupby('overall_status')['duration_years'].quantile([0.5, 0.99]).unstack())
cols = ['nct_id',
        'overall_status',
        'phase',
        'start_date',
        'start_date_type',
        'end_date',
        'end_date_type',
        'duration_years',
        'sponsor_name']
df.sort_values('duration_years', ascending=False)[cols].head(20)

,duration_years
count,90282.000000
mean,3.705483
std,4.172756
min,0.002740
50%,2.463014
90%,7.542466
99%,23.219178
99.9%,30.864575
max,63.128767


,0.50,0.99
overall_status,,
ACTIVE_NOT_RECRUITING,4.764384,18.840603
COMPLETED,2.498630,24.231644
ENROLLING_BY_INVITATION,1.886301,11.295671
RECRUITING,1.747945,10.674521
SUSPENDED,6.683562,25.715342
TERMINATED,2.402740,22.465753


,nct_id,overall_status,phase,start_date,start_date_type,end_date,end_date_type,duration_years,sponsor_name
11633,NCT00257140,COMPLETED,PHASE2/PHASE3,1931-06-30,NaN,1994-07-31,ACTUAL,63.128767,Johnson & Johnson Pharmaceutical Research & De...
2052,NCT00004767,COMPLETED,PHASE2,1985-01-31,NaN,2026-09-12,MISSING,41.641096,National Center for Research Resources (NCRR)
294,NCT00000525,COMPLETED,PHASE3,1986-07-31,NaN,2026-09-12,MISSING,40.145205,"University of California, San Francisco"
37994,NCT01324063,COMPLETED,PHASE3,1986-11-30,NaN,2026-09-12,MISSING,39.810959,European Organisation for Research and Treatme...
879,NCT00002459,COMPLETED,PHASE3,1988-04-30,NaN,2026-09-12,MISSING,38.394521,European Organisation for Research and Treatme...
16126,NCT00394511,COMPLETED,PHASE3,1988-08-31,NaN,2026-09-12,MISSING,38.057534,National Cancer Institute (NCI)
40914,NCT01468883,COMPLETED,PHASE3,1979-09-04,ACTUAL,2016-11-17,ACTUAL,37.230137,National Cancer Institute (NCI)
882,NCT00002462,ACTIVE_NOT_RECRUITING,PHASE3,1989-09-30,ACTUAL,2026-09-12,SNAPSHOT,36.975342,European Organisation for Research and Treatme...
2044,NCT00004748,COMPLETED,PHASE3,1989-11-30,NaN,2026-09-12,MISSING,36.808219,National Center for Research Resources (NCRR)
53070,NCT02177370,TERMINATED,PHASE3,1990-02-28,NaN,2026-09-12,MISSING,36.561644,Boehringer Ingelheim


# Apply the Cutoff


In [ ]:
max_years = None
if max_years is not None:
  df = exclude(df, df['duration_years'] > max_years, 7, f'Duration over {max_years}')

In [ ]:
for cutoff in [10, 15, 20, 25, 30, 35, 40]:
  n = (df['duration_years'] > cutoff).sum
  print(f'Duration over {cutoff}: {n():,}')

Duration over 10: 5,135
Duration over 15: 2,781
Duration over 20: 1,684
Duration over 25: 588
Duration over 30: 116
Duration over 35: 12
Duration over 40: 2


# Left Truncation & Trials Registered after They Ended

Entry years: how many years after the trial started did it first become visible on ClinicalTrials.gov? <br>
<br>
3 cases of entry years:
1. study_first_submitted_date = start_date
2. study_first_submitted_date < start_date
3. study_first_submitted_date > start_date

entry_years: new column<br>
reg_after_end: a column that answers if a trial was registered on/after its end date --> 7204 trials<br>
<br>
exclude function to remove the 7204 rows


In [ ]:
df['entry_years'] = ((df['study_first_submitted_date'] - df['start_date']).dt.days / 365).clip(lower=0)

reg_after_end = df['study_first_submitted_date'] >= df['end_date']
print(f'Registered on/after end date: {reg_after_end.sum():,}')

display(df.loc[reg_after_end, 'overall_status'].value_counts())

df = exclude(df,
             reg_after_end,
             8,
             'Registered on or after end date (no at-risk time under left truncation)')

Registered on/after end date: 7,203


,count
overall_status,
COMPLETED,6811
TERMINATED,392


# Outcome Coding and Grouped Features

1: terminated (event)<br>
2: completed (competing)<br>
0: censored at snapshot<br>
<br>
outcome: dictionary<br>
.map(outcome): goes down the overall_status column and replaces each status with its code <br>
<br>
end_date_estimated: terminated and completed trials that have an end_date_type != ACTUAL<br>
- True: 3447 trials: end_date_type is ESTIMATED or MISSING
- These cannot be trusted

In [ ]:
outcome = {'TERMINATED': 1,
           'COMPLETED': 2,
           'RECRUITING': 0,
           'ACTIVE_NOT_RECRUITING': 0,
           'ENROLLING_BY_INVITATION': 0,
           'SUSPENDED': 0}
df['event'] = df['overall_status'].map(outcome)
df['time_years'] = df['duration_years'].copy()
df['end_date_estimated'] = df['end_date_type'].ne('ACTUAL') & df['event'].isin([1, 2])

sponsor = {'INDUSTRY': 'Industry',
           'NIH': 'Government',
           'FED': 'Government',
           'OTHER_GOV': 'Government',
           'OTHER': 'Academic/Other',
           'NETWORK': 'Academic/Other'}
df['sponsor_type'] = df['agency_class'].map(sponsor)

df['start_year'] = df['start_date'].dt.year

area_counts = df['therapeutic_area'].value_counts()
small = area_counts[area_counts < 500].index
df['area_grouped'] = df['therapeutic_area'].where(~df['therapeutic_area'].isin(small), 'Other')

display(df['event'].value_counts())
display(pd.crosstab(df['phase'], df['event'], normalize = 'index').round(3))
df['end_date_estimated'].value_counts()

,count
event,
2,55025
0,17208
1,10846


event,0,1,2
phase,,,
PHASE2,0.223,0.146,0.631
PHASE2/PHASE3,0.218,0.131,0.651
PHASE3,0.181,0.106,0.713


,count
end_date_estimated,
False,79632
True,3447


In [ ]:
unmapped_sponsors = df[df['sponsor_type'].isna()]['agency_class'].value_counts(dropna=False)
display(unmapped_sponsors)

,count
agency_class,


In [ ]:
assert df['event'].notna().all(), 'unmapped status'
assert df['sponsor_type'].notna().all(), 'unmapped sponsor class'
assert (df['time_years'] > 0).all()
assert (df['entry_years'] < df['time_years']).all(), 'entry must precede exit'
assert (df.loc[df['event'] == 0, 'end_date'] == snapshot).all(), 'censored trials must end at snapshot'
assert log[-1]["n_remaining"] == len(df), "log out of sync with data"
assert (df.loc[df['event'].isin([1, 2]), 'end_date'] <= snapshot).all(), 'finished trials must end by snapshot'
print('All checks passed')

All checks passed
